# ⚡ Khởi Chạy Kaggle GPU Worker cho Mode `gpu_finetune` (7 Mô Hình SOTA)
**Tự động tải 7 mô hình từ Google Drive vào 2x GPU T4 & Xuất duy nhất 1 link Cloudflare Tunnel để kết nối với Web/CLI cục bộ.**

### Phân Bổ 7 Mô Hình Trên Kaggle Dual Tesla T4 (32GB VRAM):
- **GPU 0 (16GB VRAM)**: Qwen2.5-7B-Instruct (4-bit NF4) nạp đồng thời 4 LoRA Adapters:
  + `router` (Agent 1: 01_agent1_router)
  + `generator` (Agent 3: 03_agent3_summarizer/generator_qlora)
  + `critic` (Agent 3: 03_agent3_summarizer/critic_qlora)
  + `writer` (Agent 5: 05_agent5_writer DoRA+rsLoRA)
- **GPU 1 (16GB VRAM)**:
  + `factcheck` (Agent 4: 04_agent4_fact_checker BGE-Reranker-v2-m3 NLI Head, F1=1.0)
  + `reader` (Agent 2: 02_agent2_reader LayoutLMv3 Base 11-Class 2D Document Layout)
  + `embedding` (06_embedding_bge_m3 BGE-M3 MoAI LoRA)
  + `reranker` (07_reranker_bge_large CoREB LoRA)

In [ ]:
# BƯỚC 1: TỰ ĐỘNG CLONE DỰ ÁN & CÀI ĐẶT THƯ VIỆN
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    print(f"[1/3] Đang clone repo từ {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"[1/3] Repo đã tồn tại. Đang cập nhật code mới nhất...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

print("[2/3] Cài đặt các thư viện cần thiết...")
subprocess.run([
    "pip", "install", "-q",
    "fastapi", "uvicorn", "pydantic",
    "transformers", "peft", "accelerate", "bitsandbytes",
    "sentence-transformers", "gdown", "pycloudflared", "httpx"
], check=True)

print("[OK] Môi trường Dual T4 đã sẵn sàng!")

In [ ]:
# BƯỚC 2: TỰ ĐỘNG TẢI 7 MÔ HÌNH (final_weights) TỪ GOOGLE DRIVE
import gdown, zipfile, os

# Thư mục Google Drive bạn cung cấp:
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1UPF13BNjxFFhd8YFv5PpuyfOR4G-VNte?usp=sharing"
DRIVE_FOLDER_ID = "1UPF13BNjxFFhd8YFv5PpuyfOR4G-VNte"
DRIVE_FILE_ID = ""  # (Tùy chọn) Nếu bạn có File ID trực tiếp của final_weights_bundle.zip, điền vào đây

TARGET_DIR = os.path.join(PROJECT_DIR, "training_lab", "final_weights")
os.makedirs(TARGET_DIR, exist_ok=True)
zip_dest = "/kaggle/working/final_weights_bundle.zip"

# Kiểm tra nếu chưa tải thì tải từ Drive
if not os.path.exists(os.path.join(TARGET_DIR, "01_agent1_router")):
    print("[Drive Loader] Đang tải final_weights từ Google Drive...")
    if DRIVE_FILE_ID:
        gdown.download(id=DRIVE_FILE_ID, output=zip_dest, quiet=False)
    else:
        gdown.download_folder(id=DRIVE_FOLDER_ID, output="/kaggle/working/drive_download", quiet=False)
        # Tìm file zip vừa tải về
        for root, dirs, files in os.walk("/kaggle/working/drive_download"):
            for f in files:
                if f.endswith(".zip"):
                    zip_dest = os.path.join(root, f)
                    break
    if os.path.exists(zip_dest):
        print(f"[Drive Loader] Đang giải nén {zip_dest} vào {TARGET_DIR}...")
        with zipfile.ZipFile(zip_dest, "r") as zf:
            zf.extractall(TARGET_DIR)
        print("[OK] Đã giải nén thành công toàn bộ 7 mô hình!")
else:
    print("[OK] Các mô hình đã có sẵn trong training_lab/final_weights/")

print("Danh sách mô hình đã nạp:", os.listdir(TARGET_DIR))

In [ ]:
# BƯỚC 3: KHỞI ĐỘNG GPU SERVER & PHÁT LINK CLOUDFLARE TUNNEL
import subprocess, time, urllib.request, json

SERVER_SCRIPT = os.path.join(PROJECT_DIR, "kaggle", "kaggle_finetune_server.py")

print("[1/3] Khởi động FastAPI Server trên cổng 8000...")
server_process = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "kaggle.kaggle_finetune_server:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--app-dir", PROJECT_DIR
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("[2/3] Đang nạp mô hình vào Dual Tesla T4 (khoảng 30-45 giây)...")
for _ in range(60):
    try:
        res = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        if res.status == 200:
            data = json.loads(res.read())
            print("[OK] FastAPI Server đã online! Status:", data)
            break
    except Exception:
        time.sleep(2)

print("[3/3] Đang mở Cloudflare Tunnel...")
from pycloudflared import try_cloudflare
tunnel = try_cloudflare(port=8000)
tunnel_url = tunnel.tunnel.rstrip("/")

print("\n" + "="*70)
print("🚀 LINK REMOTE_INFERENCE_URL CHO MULTI-AGENT RAG (COPY DÒNG DƯỚI):")
print(f"\n    REMOTE_INFERENCE_URL={tunnel_url}\n")
print("="*70)
print("Hướng dẫn: Copy đường link trên dán vào file .env trên máy của bạn,")
print("sau đó chạy: python main/app.py --mode gpu_finetune")
print("="*70)